# Curriculum Learning — Data Size Ablation v2 (10%, 25%, 50%, 100%)

## Cell 1 — Config

In [ ]:
import os, time, random, glob, warnings, pickle, gc
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.amp import autocast, GradScaler
import timm
from torchvision import transforms
from torch.utils.data import DataLoader, Subset, Dataset
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import f1_score, confusion_matrix, classification_report
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')
%matplotlib inline

DATA_DIR     = r"C:\Users\murat akkaya\Desktop\colon_4_class"
SWIN_WEIGHTS = r"C:\Users\murat akkaya\Desktop\swin_base_pretrained.pth"
REPORT_DIR   = r"E:\CL_DataSize_Ablation_v2"
HARDNESS_CSV = r"C:\Users\murat akkaya\Desktop\Progress report\Trained Resnet\hardness_scores.csv"

N_SPLITS     = 5
BATCH_SIZE   = 64
NUM_EPOCHS   = 30
LR           = 1e-4
WEIGHT_DECAY = 1e-4
SEED         = 42
NUM_WORKERS  = 0
EMBED_DIM    = 1024
USE_AMP      = True
IMG_SIZE     = 224
CACHE_IMAGES = True

CL_START_RATIO = 0.33
CL_RAMP_EPOCHS = 25

DATA_FRACTIONS = [0.10, 0.25, 0.50, 1.00]

SAVE_FOLDS      = [1]
SAVE_LAST_MODEL = True

CLASSES = ["adenomatous", "cancer", "inflamed", "normal"]
NUM_CLASSES = len(CLASSES)

os.makedirs(REPORT_DIR, exist_ok=True)
os.makedirs(os.path.join(REPORT_DIR, 'models'), exist_ok=True)
os.makedirs(os.path.join(REPORT_DIR, 'histories'), exist_ok=True)
os.makedirs(os.path.join(REPORT_DIR, 'reports'), exist_ok=True)

torch.manual_seed(SEED); random.seed(SEED); np.random.seed(SEED)
torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision('high')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"PyTorch : {torch.__version__}")
print(f"Device  : {device}")
if device.type == 'cuda':
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
print(f"Fractions: {DATA_FRACTIONS}")

## Cell 2 — Dataset (with RAM cache)

In [ ]:
class HistopathDataset(Dataset):
    def __init__(self, root_dir, classes, transform=None, cache=True, verbose=True):
        self.transform = transform
        self.cache = cache
        self._cache = {}
        self.samples = []
        self.class_to_idx = {c: i for i, c in enumerate(classes)}
        for cls in classes:
            cls_dir = os.path.join(root_dir, cls)
            if not os.path.isdir(cls_dir): continue
            for ext in ['*.png', '*.jpg', '*.jpeg', '*.tif', '*.tiff', '*.bmp']:
                for p in sorted(glob.glob(os.path.join(cls_dir, ext))):
                    self.samples.append((p, self.class_to_idx[cls]))
        if verbose:
            print(f"Loaded {len(self.samples)} samples")
            for cls in classes:
                n = sum(1 for _, l in self.samples if l == self.class_to_idx[cls])
                print(f"  {cls}: {n}")

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        if self.cache and idx in self._cache:
            img = self._cache[idx]
        else:
            img = Image.open(path).convert('RGB').resize((IMG_SIZE, IMG_SIZE))
            if self.cache:
                self._cache[idx] = img
        if self.transform:
            img = self.transform(img)
        return img, label

train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_dataset = HistopathDataset(DATA_DIR, CLASSES, train_transform, cache=CACHE_IMAGES)
eval_dataset  = HistopathDataset(DATA_DIR, CLASSES, eval_transform, cache=CACHE_IMAGES, verbose=False)

if CACHE_IMAGES:
    from tqdm.notebook import tqdm
    print("\nCaching train images...")
    for i in tqdm(range(len(train_dataset))): train_dataset[i]
    print("Caching eval images...")
    for i in tqdm(range(len(eval_dataset))): eval_dataset[i]
    print("Cache ready.")

all_indices = np.arange(len(train_dataset))
all_targets = np.array([s[1] for s in train_dataset.samples])
all_names   = [os.path.basename(s[0]) for s in train_dataset.samples]
N_FULL = len(train_dataset)
print(f"\nFull dataset: {N_FULL}")

## Cell 3 — Hardness scores & strategies

In [ ]:
def load_hardness(csv_path, column_name, all_names):
    df = pd.read_csv(csv_path)
    df['base'] = df['filename'].astype(str).apply(lambda x: os.path.splitext(x)[0])
    hardness_map = dict(zip(df['base'], df[column_name]))
    median_val = df[column_name].median()
    scores = np.array([hardness_map.get(os.path.splitext(n)[0], median_val) for n in all_names])
    matched = sum(1 for n in all_names if os.path.splitext(n)[0] in hardness_map)
    print(f"  Matched: {matched}/{len(all_names)} | Mean: {scores.mean():.4f}")
    return scores

print("Loss:");     hardness_loss     = load_hardness(HARDNESS_CSV, 'hardness_loss', all_names)
print("Boundary:"); hardness_boundary = load_hardness(HARDNESS_CSV, 'hardness_boundary', all_names)
print("Centroid:"); hardness_centroid = load_hardness(HARDNESS_CSV, 'hardness_centroid', all_names)
print("Random:")
np.random.seed(SEED)
hardness_random = np.random.rand(N_FULL)
print(f"  Generated {N_FULL} scores")

HARDNESS_SCORES = {
    'Loss': hardness_loss, 'Boundary': hardness_boundary,
    'Centroid': hardness_centroid, 'Random': hardness_random,
}

STRATEGIES = {}
for h_name in ['Loss', 'Boundary', 'Centroid']:
    STRATEGIES[f'{h_name}_E2H'] = {'hardness': h_name, 'direction': 'e2h'}
    STRATEGIES[f'{h_name}_H2E'] = {'hardness': h_name, 'direction': 'h2e'}
STRATEGIES['Random'] = {'hardness': 'Random', 'direction': 'e2h'}

print(f"\n{len(STRATEGIES)} strategies:")
for name, cfg in STRATEGIES.items():
    d = 'Easy->Hard' if cfg['direction'] == 'e2h' else 'Hard->Easy'
    d = 'Random order' if name == 'Random' else d
    print(f"  {name:16s} {d}")

## Cell 4 — Stratified subsets + storage estimate

In [ ]:
def get_stratified_subset(all_indices, all_targets, fraction, seed=42):
    if fraction >= 1.0:
        subset_idx = all_indices.copy()
    else:
        _, subset_idx = train_test_split(
            all_indices, test_size=fraction,
            stratify=all_targets, random_state=seed
        )
        subset_idx = np.sort(subset_idx)

    subset_labels = all_targets[subset_idx]
    print(f"  Subset: {len(subset_idx)} samples ({fraction*100:.0f}%)")
    for cls_idx, cls_name in enumerate(CLASSES):
        print(f"    {cls_name}: {(subset_labels == cls_idx).sum()}")
    return subset_idx

SUBSETS = {}
for frac in DATA_FRACTIONS:
    print(f"\n--- {frac*100:.0f}% subset ---")
    SUBSETS[frac] = get_stratified_subset(all_indices, all_targets, frac, SEED)

n_ckpt_per_run = 1 + (1 if SAVE_LAST_MODEL else 0)
n_models = len(STRATEGIES) * len(DATA_FRACTIONS) * len(SAVE_FOLDS) * n_ckpt_per_run
est_gb = n_models * 0.35
n_runs = len(STRATEGIES) * len(DATA_FRACTIONS) * N_SPLITS

print(f"\n{'='*55}")
print(f"  Training runs : {n_runs}  ({len(STRATEGIES)} strategies x {len(DATA_FRACTIONS)} fractions x {N_SPLITS} folds)")
print(f"  Models saved  : {n_models}  (folds {SAVE_FOLDS}, last_model={SAVE_LAST_MODEL})")
print(f"  Est. storage  : ~{est_gb:.1f} GB")
print(f"{'='*55}")
if est_gb > 40:
    print("  WARNING: large. Reduce SAVE_FOLDS or set SAVE_LAST_MODEL=False.")

## Cell 5 — Model, curriculum, validation

In [ ]:
if not os.path.exists(SWIN_WEIGHTS):
    _tmp = timm.create_model('swin_base_patch4_window7_224', pretrained=True, num_classes=0)
    torch.save(_tmp.state_dict(), SWIN_WEIGHTS)
    del _tmp; torch.cuda.empty_cache()

def create_model():
    backbone = timm.create_model('swin_base_patch4_window7_224', pretrained=False, num_classes=0)
    backbone.load_state_dict(torch.load(SWIN_WEIGHTS, weights_only=True))
    return nn.Sequential(backbone, nn.Linear(EMBED_DIM, NUM_CLASSES)).to(device)

def get_curriculum_indices(train_idx, hardness_scores, epoch, direction='e2h'):
    progress = min(epoch / CL_RAMP_EPOCHS, 1.0)
    ratio = min(CL_START_RATIO + (1.0 - CL_START_RATIO) * progress, 1.0)
    hardness_subset = hardness_scores[train_idx]
    if direction == 'e2h':
        sorted_order = np.argsort(hardness_subset)
    else:
        sorted_order = np.argsort(hardness_subset)[::-1]
    n_samples = max(int(len(train_idx) * ratio), min(BATCH_SIZE, len(train_idx)))
    return train_idx[sorted_order[:n_samples]], n_samples

def validate(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0.0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            with autocast('cuda', enabled=USE_AMP):
                out = model(imgs)
                total_loss += F.cross_entropy(out, labels).item()
            all_preds.extend(out.argmax(1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    avg_loss = total_loss / len(loader)
    acc = 100 * np.mean(np.array(all_preds) == np.array(all_labels))
    macro_f1 = f1_score(all_labels, all_preds, average='macro') * 100
    return avg_loss, acc, macro_f1, np.array(all_preds), np.array(all_labels)

def save_checkpoint(state_dict, strategy, frac, fold, epoch, val_f1, tag,
                    train_idx, test_idx, subset_idx):
    frac_tag = f"frac{int(frac*100):03d}"
    fname = f"{strategy}_{frac_tag}_fold{fold}_{tag}.pth"
    path = os.path.join(REPORT_DIR, 'models', fname)
    torch.save({
        'state_dict': state_dict,
        'strategy': strategy, 'fraction': frac, 'fold': fold,
        'epoch': epoch, 'val_f1': val_f1, 'tag': tag,
        'class_names': CLASSES,
        'train_idx': np.asarray(train_idx),
        'test_idx': np.asarray(test_idx),
        'subset_idx': np.asarray(subset_idx),
        'img_size': IMG_SIZE, 'embed_dim': EMBED_DIM,
        'arch': 'swin_base_patch4_window7_224',
    }, path)
    return path

print("Ready.")

## Cell 6 — Training loop

In [ ]:
def train_fold(strategy_name, hardness_scores, direction,
               train_idx, test_idx, subset_idx, fold, frac, frac_label):
    model = create_model()
    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)
    scaler = GradScaler('cuda', enabled=USE_AMP)

    test_loader = DataLoader(Subset(eval_dataset, test_idx), batch_size=BATCH_SIZE,
                             shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

    n_total = len(train_idx)
    best_f1, best_epoch, best_time, best_state = 0.0, 0, 0.0, None
    saved_paths = {}
    history = {'train_loss': [], 'train_f1': [], 'val_loss': [], 'val_f1': [],
               'val_acc': [], 'samples_used': []}
    do_save = fold in SAVE_FOLDS
    start_time = time.time()

    for epoch in range(1, NUM_EPOCHS + 1):
        cl_idx, n_used = get_curriculum_indices(train_idx, hardness_scores, epoch, direction)
        train_loader = DataLoader(Subset(train_dataset, cl_idx), batch_size=BATCH_SIZE,
                                  shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)

        model.train()
        run_loss = 0.0
        ep_preds, ep_labels = [], []
        for imgs, labels in train_loader:
            imgs = imgs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with autocast('cuda', enabled=USE_AMP):
                out = model(imgs)
                loss = F.cross_entropy(out, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            run_loss += loss.item()
            ep_preds.extend(out.argmax(1).detach().cpu().numpy())
            ep_labels.extend(labels.detach().cpu().numpy())
        scheduler.step()

        train_loss = run_loss / len(train_loader)
        train_f1 = f1_score(ep_labels, ep_preds, average='macro') * 100
        val_loss, val_acc, val_f1, _, _ = validate(model, test_loader)

        history['train_loss'].append(train_loss)
        history['train_f1'].append(train_f1)
        history['val_loss'].append(val_loss)
        history['val_f1'].append(val_f1)
        history['val_acc'].append(val_acc)
        history['samples_used'].append(n_used)

        if val_f1 > best_f1:
            best_f1, best_epoch = val_f1, epoch
            best_time = time.time() - start_time
            if do_save:
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        if epoch % 10 == 0 or epoch == 1:
            print(f"  [{frac_label}] [{strategy_name}] F{fold} Ep{epoch:2d} | "
                  f"CL: {n_used}/{n_total} | TF1: {train_f1:.1f}% | VF1: {val_f1:.1f}%")

    total_time = time.time() - start_time

    _, last_acc, last_f1, last_preds, last_labels = validate(model, test_loader)
    last_per_class = (f1_score(last_labels, last_preds, average=None) * 100).tolist()
    last_cm = confusion_matrix(last_labels, last_preds)

    if do_save:
        if best_state is not None:
            saved_paths['best'] = save_checkpoint(
                best_state, strategy_name, frac, fold, best_epoch, best_f1,
                'best', train_idx, test_idx, subset_idx)
            print(f"    >>> saved best  (ep{best_epoch}, VF1={best_f1:.2f}%)")
        if SAVE_LAST_MODEL:
            last_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            saved_paths['last'] = save_checkpoint(
                last_state, strategy_name, frac, fold, NUM_EPOCHS, last_f1,
                'last', train_idx, test_idx, subset_idx)
            print(f"    >>> saved last  (ep{NUM_EPOCHS}, VF1={last_f1:.2f}%)")
            del last_state

        rep = classification_report(last_labels, last_preds, target_names=CLASSES, zero_division=0)
        rname = f"{strategy_name}_frac{int(frac*100):03d}_fold{fold}.txt"
        with open(os.path.join(REPORT_DIR, 'reports', rname), 'w') as f:
            f.write(f"Strategy: {strategy_name}\nFraction: {frac}\nFold: {fold}\n")
            f.write(f"Best epoch: {best_epoch} (F1 {best_f1:.2f}%)\n")
            f.write(f"Last epoch: {NUM_EPOCHS} (F1 {last_f1:.2f}%)\n\n")
            f.write(rep)

    print(f"  Best: ep{best_epoch} VF1={best_f1:.2f}% ({best_time:.0f}s) | "
          f"Last: VF1={last_f1:.2f}% ({total_time:.0f}s)")

    del model, optimizer, scheduler, scaler, best_state
    torch.cuda.empty_cache(); gc.collect()

    return {'history': history, 'best_f1': best_f1, 'best_epoch': best_epoch,
            'best_time': best_time, 'last_f1': last_f1, 'last_acc': last_acc,
            'last_per_class': last_per_class, 'last_cm': last_cm,
            'total_time': total_time, 'saved_paths': saved_paths}

## Cell 7 — Run all fractions × strategies

In [ ]:
strategies_to_run = list(STRATEGIES.keys())
all_fraction_results = {}
grand_start = time.time()

for frac in DATA_FRACTIONS:
    frac_label = f"{frac*100:.0f}%"
    subset_idx = SUBSETS[frac]
    subset_targets = all_targets[subset_idx]

    print(f"\n{'#'*70}")
    print(f"  DATA FRACTION: {frac_label} ({len(subset_idx)} samples)")
    print(f"{'#'*70}")

    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    folds = list(skf.split(np.zeros(len(subset_idx)), subset_targets))

    frac_results = {}
    for strategy_name in strategies_to_run:
        cfg = STRATEGIES[strategy_name]
        h_scores = HARDNESS_SCORES[cfg['hardness']]
        direction = cfg['direction']

        print(f"\n{'='*60}")
        print(f"  {frac_label} | {strategy_name}")
        print(f"{'='*60}")

        strat = {'best_f1s': [], 'last_f1s': [], 'last_accs': [], 'histories': [],
                 'best_epochs': [], 'best_times': [], 'total_times': [],
                 'last_per_class': [], 'last_cms': [], 'saved_paths': []}

        for fold_idx, (f_train, f_test) in enumerate(folds):
            fold = fold_idx + 1
            real_train_idx = subset_idx[f_train]
            real_test_idx  = subset_idx[f_test]

            res = train_fold(strategy_name, h_scores, direction,
                             real_train_idx, real_test_idx, subset_idx,
                             fold, frac, frac_label)

            strat['best_f1s'].append(res['best_f1'])
            strat['last_f1s'].append(res['last_f1'])
            strat['last_accs'].append(res['last_acc'])
            strat['histories'].append(res['history'])
            strat['best_epochs'].append(res['best_epoch'])
            strat['best_times'].append(res['best_time'])
            strat['total_times'].append(res['total_time'])
            strat['last_per_class'].append(res['last_per_class'])
            strat['last_cms'].append(res['last_cm'])
            strat['saved_paths'].append(res['saved_paths'])

        print(f"  [{frac_label}] [{strategy_name}] Best F1="
              f"{np.mean(strat['best_f1s']):.2f}+/-{np.std(strat['best_f1s']):.2f}%")
        frac_results[strategy_name] = strat

    all_fraction_results[frac] = frac_results

print(f"\nAll done in {(time.time()-grand_start)/3600:.1f} hours")

## Cell 8 — Save histories

In [ ]:
pkl_path = os.path.join(REPORT_DIR, 'histories', 'datasize_ablation_v2.pkl')
with open(pkl_path, 'wb') as f:
    pickle.dump(all_fraction_results, f)

rows = []
for frac, frac_res in all_fraction_results.items():
    for strat_name, strat in frac_res.items():
        cfg = STRATEGIES[strat_name]
        for fi, hist in enumerate(strat['histories']):
            for ep in range(len(hist['train_loss'])):
                rows.append({
                    'fraction': frac, 'fraction_label': f"{frac*100:.0f}%",
                    'n_total_samples': len(SUBSETS[frac]),
                    'strategy': strat_name, 'hardness': cfg['hardness'],
                    'direction': cfg['direction'],
                    'fold': fi + 1, 'epoch': ep + 1,
                    'train_loss': hist['train_loss'][ep], 'train_f1': hist['train_f1'][ep],
                    'val_loss': hist['val_loss'][ep], 'val_f1': hist['val_f1'][ep],
                    'val_acc': hist['val_acc'][ep], 'samples_used': hist['samples_used'][ep],
                })
epoch_csv = os.path.join(REPORT_DIR, 'histories', 'datasize_epoch_data.csv')
pd.DataFrame(rows).to_csv(epoch_csv, index=False)

srows = []
for frac, frac_res in all_fraction_results.items():
    for strat_name, strat in frac_res.items():
        cfg = STRATEGIES[strat_name]
        for fi in range(N_SPLITS):
            pc = strat['last_per_class'][fi]
            srows.append({
                'fraction': frac, 'fraction_label': f"{frac*100:.0f}%",
                'n_samples': len(SUBSETS[frac]),
                'strategy': strat_name, 'hardness': cfg['hardness'],
                'direction': cfg['direction'], 'fold': fi + 1,
                'best_f1': strat['best_f1s'][fi], 'best_epoch': strat['best_epochs'][fi],
                'time_to_best_s': strat['best_times'][fi],
                'last_f1': strat['last_f1s'][fi], 'last_acc': strat['last_accs'][fi],
                'total_time_s': strat['total_times'][fi],
                'f1_adenomatous': pc[0], 'f1_cancer': pc[1],
                'f1_inflamed': pc[2], 'f1_normal': pc[3],
            })
fold_csv = os.path.join(REPORT_DIR, 'histories', 'datasize_per_fold.csv')
pd.DataFrame(srows).to_csv(fold_csv, index=False)

arows = []
for frac, frac_res in all_fraction_results.items():
    for strat_name, strat in frac_res.items():
        cfg = STRATEGIES[strat_name]
        arows.append({
            'fraction': frac, 'fraction_label': f"{frac*100:.0f}%",
            'n_samples': len(SUBSETS[frac]),
            'strategy': strat_name, 'hardness': cfg['hardness'], 'direction': cfg['direction'],
            'best_f1_mean': np.mean(strat['best_f1s']), 'best_f1_std': np.std(strat['best_f1s']),
            'last_f1_mean': np.mean(strat['last_f1s']), 'last_f1_std': np.std(strat['last_f1s']),
            'avg_best_epoch': np.mean(strat['best_epochs']),
            'avg_time_to_best': np.mean(strat['best_times']),
            'avg_total_time': np.mean(strat['total_times']),
        })
summary_csv = os.path.join(REPORT_DIR, 'histories', 'datasize_summary.csv')
summary_df = pd.DataFrame(arows)
summary_df.to_csv(summary_csv, index=False)

print(f"Saved: {pkl_path}")
print(f"Saved: {epoch_csv}")
print(f"Saved: {fold_csv}")
print(f"Saved: {summary_csv}")

## Cell 9 — Comparison table

In [ ]:
print(f"\n{'='*100}")
print(f"  DATA SIZE ABLATION - Best Model Macro F1")
print(f"{'='*100}")

header = f"{'Strategy':<16}"
for frac in DATA_FRACTIONS:
    header += f"{f'{frac*100:.0f}%':>18}"
print(header)
print('-' * 100)

for strat_name in strategies_to_run:
    line = f"{strat_name:<16}"
    for frac in DATA_FRACTIONS:
        s = all_fraction_results[frac][strat_name]
        cell = f"{np.mean(s['best_f1s']):.2f}+/-{np.std(s['best_f1s']):.2f}"
        line += f"{cell:>18}"
    print(line)
print(f"{'='*100}")

print("\nBest strategy per fraction (and margin over Random):")
for frac in DATA_FRACTIONS:
    fr = all_fraction_results[frac]
    means = {k: np.mean(v['best_f1s']) for k, v in fr.items()}
    winner = max(means, key=means.get)
    rnd = means.get('Random', np.nan)
    print(f"  {frac*100:>3.0f}%: {winner:<16} {means[winner]:.2f}%   "
          f"(Random {rnd:.2f}%, margin {means[winner]-rnd:+.2f})")

## Cell 10 — F1 vs data size (colorblind-safe)

In [ ]:
cb_colors = {
    'Loss_E2H':     '#0072B2', 'Loss_H2E':     '#56B4E9',
    'Boundary_E2H': '#D55E00', 'Boundary_H2E': '#E69F00',
    'Centroid_E2H': '#009E73', 'Centroid_H2E': '#66CC99',
    'Random':       '#000000',
}
cb_ls = {k: ('-' if k.endswith('E2H') else ('--' if k.endswith('H2E') else ':'))
         for k in cb_colors}
cb_mk = {k: ('o' if k.endswith('E2H') else ('s' if k.endswith('H2E') else 'D'))
         for k in cb_colors}

x = np.arange(len(DATA_FRACTIONS))
x_labels = [f"{f*100:.0f}%\n({len(SUBSETS[f])})" for f in DATA_FRACTIONS]

fig, ax = plt.subplots(figsize=(12, 7))
for strat_name in strategies_to_run:
    means = [np.mean(all_fraction_results[f][strat_name]['best_f1s']) for f in DATA_FRACTIONS]
    stds  = [np.std(all_fraction_results[f][strat_name]['best_f1s'])  for f in DATA_FRACTIONS]
    lw = 3 if strat_name == 'Random' else 2
    ax.errorbar(x, means, yerr=stds, marker=cb_mk[strat_name], ms=8, lw=lw, capsize=4,
                color=cb_colors[strat_name], ls=cb_ls[strat_name], label=strat_name)

ax.set_xticks(x); ax.set_xticklabels(x_labels, fontsize=11)
ax.set_xlabel('Training Data Fraction (n samples)', fontsize=13)
ax.set_ylabel('Best Macro F1 (%)', fontsize=13)
ax.set_title('Curriculum Learning Effect vs Data Size', fontsize=15, fontweight='bold')
ax.legend(fontsize=10, ncol=2); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(REPORT_DIR, 'f1_vs_datasize.png'), dpi=150, bbox_inches='tight')
plt.show()

fig, ax = plt.subplots(figsize=(12, 6))
for strat_name in strategies_to_run:
    if strat_name == 'Random': continue
    margins = [np.mean(all_fraction_results[f][strat_name]['best_f1s']) -
               np.mean(all_fraction_results[f]['Random']['best_f1s']) for f in DATA_FRACTIONS]
    ax.plot(x, margins, marker=cb_mk[strat_name], ms=8, lw=2,
            color=cb_colors[strat_name], ls=cb_ls[strat_name], label=strat_name)
ax.axhline(0, color='black', lw=1.5, ls=':', label='Random baseline')
ax.set_xticks(x); ax.set_xticklabels(x_labels, fontsize=11)
ax.set_xlabel('Training Data Fraction (n samples)', fontsize=13)
ax.set_ylabel('Macro F1 margin over Random (pp)', fontsize=13)
ax.set_title('Curriculum Benefit Relative to Random Ordering', fontsize=15, fontweight='bold')
ax.legend(fontsize=10, ncol=2); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(REPORT_DIR, 'margin_over_random.png'), dpi=150, bbox_inches='tight')
plt.show()

## Cell 11 — Validation F1 curves (all strategies × all fractions)

In [ ]:
df = pd.read_csv(os.path.join(REPORT_DIR, 'histories', 'datasize_epoch_data.csv'))
epochs = np.arange(1, NUM_EPOCHS + 1)
n_frac = len(DATA_FRACTIONS)

fig, axes = plt.subplots(1, n_frac, figsize=(6.5 * n_frac, 6), sharey=True)
if n_frac == 1: axes = [axes]

for idx, frac in enumerate(DATA_FRACTIONS):
    ax = axes[idx]
    sub = df[df['fraction'] == frac]
    for strat in strategies_to_run:
        sd = sub[sub['strategy'] == strat]
        if sd.empty: continue
        agg = sd.groupby('epoch')['val_f1'].agg(['mean', 'std'])
        lw = 3 if strat == 'Random' else 2
        ax.plot(epochs, agg['mean'], color=cb_colors[strat], ls=cb_ls[strat], lw=lw, label=strat)
        ax.fill_between(epochs, agg['mean'] - agg['std'], agg['mean'] + agg['std'],
                        color=cb_colors[strat], alpha=0.06)
    ax.set_title(f"{frac*100:.0f}% ({len(SUBSETS[frac])} samples)", fontsize=14, fontweight='bold')
    ax.set_xlabel('Epoch', fontsize=12)
    if idx == 0: ax.set_ylabel('Validation Macro F1 (%)', fontsize=12)
    ax.grid(True, alpha=0.3)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=4, fontsize=10, bbox_to_anchor=(0.5, -0.06))
plt.suptitle('Validation F1 Curves — All Strategies x Data Sizes', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(REPORT_DIR, 'val_f1_all_strategies_all_fractions.png'),
            dpi=150, bbox_inches='tight')
plt.show()

## Cell 12 — Saved models index

In [ ]:
model_files = sorted(glob.glob(os.path.join(REPORT_DIR, 'models', '*.pth')))
total_gb = sum(os.path.getsize(f) for f in model_files) / 1e9

rows = []
for f in model_files:
    ck = torch.load(f, map_location='cpu', weights_only=False)
    rows.append({
        'path': f, 'filename': os.path.basename(f),
        'strategy': ck['strategy'], 'fraction': ck['fraction'], 'fold': ck['fold'],
        'tag': ck['tag'], 'epoch': ck['epoch'], 'val_f1': ck['val_f1'],
        'n_train': len(ck['train_idx']), 'n_test': len(ck['test_idx']),
    })
    del ck

idx_df = pd.DataFrame(rows)
idx_csv = os.path.join(REPORT_DIR, 'models_index.csv')
idx_df.to_csv(idx_csv, index=False)

print(f"{len(model_files)} checkpoints, {total_gb:.1f} GB")
print(f"Index: {idx_csv}\n")
if len(idx_df):
    display(idx_df[['filename', 'strategy', 'fraction', 'fold', 'tag', 'epoch', 'val_f1']])

print("\nTo extract embeddings later:")
print("  ck = torch.load(path, map_location='cpu', weights_only=False)")
print("  backbone = timm.create_model(ck['arch'], pretrained=False, num_classes=0)")
print("  backbone.load_state_dict({k[2:]: v for k, v in ck['state_dict'].items() if k.startswith('0.')})")
print("  train_idx, test_idx = ck['train_idx'], ck['test_idx']   # exact original split")